# Demo 13 - Heston calibration versus local-vol PDE

This notebook is the final Capstone 3 comparison artifact. It keeps the demo layer thin: build the deterministic market-like synthetic fixture, calibrate Heston once with robust quadrature, rerun headline diagnostics at diagnostics quality, call `run_heston_vs_local_vol_comparison(...)`, and display the diagnostic tables without rebuilding pricing, eSSVI, Dupire, PDE, residual, bucket, or tradeoff logic in the notebook.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    src = candidate / "src"
    if (src / "option_pricing").exists():
        sys.path.insert(0, str(src))
        break

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from option_pricing.diagnostics.heston import (
    build_market_like_heston_quote_set,
    plot_heston_model_comparison_error_buckets,
    plot_heston_model_comparison_iv_residual_heatmap,
    plot_heston_model_comparison_smile_overlay,
    plot_heston_model_comparison_train_heldout,
    run_heston_calibration_fit_diagnostics,
    run_heston_vs_local_vol_comparison,
)
from option_pricing.models.heston.calibration import calibrate_heston_multistart
from option_pricing.models.heston.calibration.bounds import HestonCalibrationBounds
from option_pricing.models.heston.params import HESTON_PARAM_NAMES
from option_pricing.numerics.quadrature import QuadratureConfig
from option_pricing.vol.ssvi import ESSVICalibrationConfig, ESSVIProjectionConfig

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 88)

## Common target quote set

The target is a deterministic market-like synthetic fixture. It is not market data and is not generated from the fitted Heston model. The held-out split is defined once and passed into diagnostics so each comparison row carries the same `train` or `held_out` label.

In [ ]:
robust_quad_cfg = QuadratureConfig(u_max=220.0, n_panels=44, nodes_per_panel=24)
diagnostics_quad_cfg = QuadratureConfig(u_max=260.0, n_panels=56, nodes_per_panel=32)

quotes = build_market_like_heston_quote_set(
    expiries=np.array([0.5, 1.0, 2.0], dtype=np.float64),
    log_moneyness=np.array([-0.12, -0.06, 0.0, 0.06, 0.12], dtype=np.float64),
)

held_out_mask = np.zeros(quotes.n_quotes, dtype=np.bool_)
held_out_mask[4::5] = True

pd.DataFrame(
    {
        "quote_index": np.arange(quotes.n_quotes, dtype=np.int64),
        "expiry": quotes.expiry,
        "log_moneyness": quotes.log_moneyness,
        "strike": quotes.strike,
        "market_iv": quotes.iv_mid,
        "market_price": quotes.mid,
        "sample": np.where(held_out_mask, "held_out", "train"),
    }
)

## Heston calibration

The notebook uses the library multistart calibrator and then hands the result to the comparison diagnostic. The optimizer settings are intentionally small and deterministic for demo runtime.

In [ ]:
bounds = HestonCalibrationBounds()
heston_fit = calibrate_heston_multistart(
    quotes=quotes,
    objective_type="vega_scaled_price",
    bounds=bounds,
    quad_cfg=robust_quad_cfg,
    max_seeds=6,
    parameter_transform="bounded",
    loss="linear",
    max_nfev=100,
)

headline_diagnostics = run_heston_calibration_fit_diagnostics(
    quotes=quotes,
    fit=heston_fit,
    held_out_mask=held_out_mask,
    quad_cfg=diagnostics_quad_cfg,
    objective_slice_grid_size=2,
)

pd.DataFrame(
    {
        "parameter": HESTON_PARAM_NAMES,
        "seed": heston_fit.best_run.seed_params.as_array(),
        "fitted": heston_fit.best_params.as_array(),
        "lower_bound": bounds.lower_array(),
        "upper_bound": bounds.upper_array(),
    }
)

In [ ]:
pd.DataFrame(
    [
        {
            "seed_index": heston_fit.best_run.seed_index,
            "success": heston_fit.best_run.success,
            "cost": heston_fit.best_run.cost,
            "optimality": heston_fit.best_run.optimality,
            "nfev": heston_fit.best_run.nfev,
            "message": heston_fit.best_run.message,
        }
    ]
)

## Heston versus local-vol/eSSVI proxy

The comparison object is the source of truth for residual rows, summary metrics, held-out splits, buckets, tradeoffs, and limitation notes.

In [ ]:
comparison = run_heston_vs_local_vol_comparison(
    quotes=quotes,
    heston_fit=heston_fit,
    held_out_mask=held_out_mask,
    heston_quad_cfg=robust_quad_cfg,
    essvi_cfg=ESSVICalibrationConfig(max_nfev=800),
    essvi_projection_cfg=ESSVIProjectionConfig(
        validation_nt=21,
        validation_y_min=-0.60,
        validation_y_max=0.60,
        validation_ny=41,
        dupire_nt=15,
        dupire_y_min=-0.50,
        dupire_y_max=0.50,
        dupire_ny=31,
        strict_validation=False,
    ),
    local_vol_pde_max_quotes=9,
    local_vol_pde_Nx=81,
    local_vol_pde_Nt=121,
)

assert set(comparison.tables) >= {
    "fit_errors",
    "error_summary",
    "held_out_comparison",
    "tradeoff_summary",
    "direct_local_vol_pde",
}
assert comparison.meta["direct_local_vol_pde_repricing"] is True
assert comparison.meta["direct_local_vol_pde_success_count"] > 0

comparison.meta

## Diagnostic plots

These figures consume the packaged comparison diagnostics. The notebook does not rebuild residuals, buckets, train/held-out summaries, Heston pricing, eSSVI repricing, or direct PDE repricing data.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14.0, 9.0), constrained_layout=True)

plot_heston_model_comparison_smile_overlay(
    comparison,
    expiry=1.0,
    ax=axes[0, 0],
    title="Smile overlay at T=1.0",
)
plot_heston_model_comparison_error_buckets(
    comparison,
    metric="iv_rmse_bps",
    ax=axes[0, 1],
    title="Bucketed IV RMSE",
)
plot_heston_model_comparison_train_heldout(
    comparison,
    metric="iv_rmse_bps",
    ax=axes[1, 0],
    title="Train vs held-out IV RMSE",
)
plot_heston_model_comparison_iv_residual_heatmap(
    comparison,
    model="Heston",
    ax=axes[1, 1],
    title="IV residual heatmap",
)

fig


## Side-by-side residual heatmaps

The long-form `fit_errors` table is heatmap-ready for each model. This side-by-side view makes maturity and moneyness structure visible without any notebook-local residual logic.

In [ ]:
model_names = list(comparison.meta["models"])
fig, axes = plt.subplots(
    1,
    len(model_names),
    figsize=(6.5 * len(model_names), 4.2),
    constrained_layout=True,
)

for ax, model_name in zip(np.atleast_1d(axes), model_names, strict=True):
    plot_heston_model_comparison_iv_residual_heatmap(
        comparison,
        model=model_name,
        ax=ax,
        title="IV residual heatmap",
    )

fig


## Fit quality and held-out split

In [ ]:
comparison.tables["held_out_comparison"]

In [ ]:
comparison.tables["error_summary"]

## Heatmap-ready residual rows

The diagnostic returns quote-level residual rows in long form, including model name, expiry, log-moneyness, bucket, and sample label. A plotting notebook can pivot this table without recomputing either model.

In [ ]:
comparison.tables["fit_errors"].loc[
    :,
    [
        "model",
        "quote_index",
        "expiry",
        "log_moneyness",
        "moneyness_bucket",
        "sample",
        "market_iv",
        "model_iv",
        "iv_residual_bps",
        "price_residual",
    ],
]

## Tradeoff summary

The table below is qualitative by design. It supports the final narrative: Heston gives interpretable stochastic-volatility dynamics, eSSVI is the flexible vanilla-surface side, and direct local-vol PDE repricing audits a small Dupire validation grid.

In [ ]:
comparison.tables["tradeoff_summary"]

## Notes and limitations

These notes intentionally travel with the diagnostic metadata and should be carried into the capstone narrative.

In [ ]:
display(Markdown("\n".join(f"- {note}" for note in comparison.meta["notes"])))